# Optimized marginal timing for Tsallis and thermodynamic scanpath statistics

This notebook measures the **marginal** time required to compute the two feature families after shared scanpath representations have already been cached.

The old extraction notebook instantiated one transformer per configuration. That caused repeated calls to spatial discretization and repeated construction of state/transition counts for each `q`, `beta`, `escort_gamma`, and grid variant. Here, the expensive shared objects are computed once per scanpath and grid:

- spatial states;
- state counts and duration-weighted state counts;
- transition counts and duration-weighted transition counts;
- smoothed probability vectors;
- smoothed transition matrices and stationary distributions;
- kinematic and surprisal energy series used by thermodynamic observables.

The timing table at the end separates:

1. shared cache construction time;
2. marginal Tsallis feature time;
3. marginal thermodynamic feature time.

In [ ]:
from __future__ import annotations

import gc
import math
import sys
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable, Optional

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
DATASETS_DIR = Path("data")
DATASET_NAME: str | None = None      # e.g. "my_dataset_fixations" without .csv
DATASET_PATH: str | Path | None = None

# Use all scanpaths by default. Set an integer for a quick pilot, e.g. 100.
MAX_SCANPATHS: int | None = None
SORT_WITHIN_SCANPATH = False

GRID_SIZES = (2, 4, 6, 8, 10)

SMOOTHING = 0.5
BASE = 2.0

FINE_QS = (0.1, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0, 4.0, 5.0)
FINE_BETAS = (0.1, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0, 4.0, 5.0)
Q_ENSEMBLES_FINE = (1.5, 1.0, 0.5)

# Number of repeated timing runs after cache construction.
N_REPEATS = 5

# Save computed feature matrices and timing summary.
SAVE_OUTPUTS = True
OUTPUT_DIR = Path("timing_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
ALLOW_SYNTHETIC_FALLBACK = True

## Dataset loading

In [ ]:
def _infer_columns_fallback(df: pd.DataFrame) -> dict[str, Any]:
    """Minimal fallback if src.utils.dataset_utils is unavailable."""
    group_cols = [c for c in df.columns if c.startswith("group_")]
    if "norm_pos_x" in df.columns and "norm_pos_y" in df.columns:
        x_col, y_col = "norm_pos_x", "norm_pos_y"
    elif "x" in df.columns and "y" in df.columns:
        x_col, y_col = "x", "y"
        # Normalize to match the project loader convention.
        max_x = pd.to_numeric(df[x_col], errors="coerce").max()
        max_y = pd.to_numeric(df[y_col], errors="coerce").max()
        if pd.notna(max_x) and max_x > 1:
            df["norm_pos_x"] = pd.to_numeric(df[x_col], errors="coerce") / max_x
            x_col = "norm_pos_x"
        if pd.notna(max_y) and max_y > 1:
            df["norm_pos_y"] = pd.to_numeric(df[y_col], errors="coerce") / max_y
            y_col = "norm_pos_y"
    else:
        raise ValueError("Could not infer x/y columns. Expected norm_pos_x/norm_pos_y or x/y.")

    if "timestamp" in df.columns:
        t_col = "timestamp"
    elif "timestamp_start" in df.columns:
        t_col = "timestamp_start"
    elif "time" in df.columns:
        t_col = "time"
    else:
        t_col = None

    if "duration" in df.columns:
        duration_col = "duration"
    elif "timestamp_start" in df.columns and "timestamp_end" in df.columns:
        df["duration"] = pd.to_numeric(df["timestamp_end"], errors="coerce") - pd.to_numeric(df["timestamp_start"], errors="coerce")
        duration_col = "duration"
    else:
        duration_col = None

    return {
        "group_cols": group_cols,
        "x_col": x_col,
        "y_col": y_col,
        "t_col": t_col,
        "duration_col": duration_col,
        "aoi_col": None,
    }


def _make_synthetic_dataset(n_subjects: int = 6, n_items: int = 4, n_fix: int = 50, seed: int = 42):
    rng = np.random.default_rng(seed)
    rows = []
    for s in range(n_subjects):
        for item in range(n_items):
            x = np.clip(np.cumsum(rng.normal(0, 0.06, n_fix)) + rng.uniform(0.25, 0.75), 0, 1)
            y = np.clip(np.cumsum(rng.normal(0, 0.06, n_fix)) + rng.uniform(0.25, 0.75), 0, 1)
            durations = rng.gamma(shape=2.0, scale=80.0, size=n_fix) + 40.0
            timestamp = np.cumsum(durations)
            for k in range(n_fix):
                rows.append({
                    "group_subject": f"S{s:02d}",
                    "group_item": f"I{item:02d}",
                    "norm_pos_x": x[k],
                    "norm_pos_y": y[k],
                    "timestamp": timestamp[k],
                    "duration": durations[k],
                })
    df = pd.DataFrame(rows)
    col_info = {
        "group_cols": ["group_subject", "group_item"],
        "x_col": "norm_pos_x",
        "y_col": "norm_pos_y",
        "t_col": "timestamp",
        "duration_col": "duration",
        "aoi_col": None,
    }
    return df, col_info, "synthetic_scanpaths"


def load_single_dataset() -> tuple[pd.DataFrame, dict[str, Any], str]:
    if DATASET_PATH is not None:
        path = Path(DATASET_PATH)
        if not path.exists():
            raise FileNotFoundError(path)
        try:
            from src.utils.dataset_utils import load_and_preprocess_dataset
            df, col_info, _ = load_and_preprocess_dataset(path)
        except Exception:
            df = pd.read_csv(path)
            col_info = _infer_columns_fallback(df)
        dataset_name = path.stem
    else:
        if DATASETS_DIR.exists():
            candidates = []
            if DATASET_NAME is not None:
                p = DATASETS_DIR / f"{DATASET_NAME}.csv"
                if p.exists():
                    candidates = [p]
                else:
                    candidates = sorted(DATASETS_DIR.glob(f"{DATASET_NAME}*.csv"))
            else:
                try:
                    from src.utils.dataset_utils import find_all_datasets
                    all_datasets = find_all_datasets(DATASETS_DIR)
                    candidates = list(all_datasets.get("fixation", [])) + list(all_datasets.get("unknown", []))
                except Exception:
                    candidates = sorted(DATASETS_DIR.glob("*.csv"))
            candidates = [p for p in candidates if p.name != "Datasets_description.csv"]
        else:
            candidates = []

        if not candidates:
            if not ALLOW_SYNTHETIC_FALLBACK:
                raise FileNotFoundError(f"No CSV datasets found in {DATASETS_DIR.resolve()}")
            warnings.warn("No real dataset found; using a synthetic fallback. Use real data for final timing.")
            return _make_synthetic_dataset()

        path = candidates[0]
        try:
            from src.utils.dataset_utils import load_and_preprocess_dataset
            df, col_info, _ = load_and_preprocess_dataset(path)
            if df is None or len(df) == 0:
                raise ValueError(f"Dataset loader returned empty dataframe for {path}")
        except Exception:
            df = pd.read_csv(path)
            col_info = _infer_columns_fallback(df)
        dataset_name = path.stem

    if "duration" in df.columns and not col_info.get("duration_col"):
        col_info["duration_col"] = "duration"
    col_info.setdefault("aoi_col", None)
    col_info.setdefault("group_cols", [c for c in df.columns if c.startswith("group_")])

    return df, col_info, dataset_name


df_raw, col_info, dataset_name = load_single_dataset()

x_col = col_info.get("x_col")
y_col = col_info.get("y_col")
t_col = col_info.get("t_col")
duration_col = col_info.get("duration_col")
group_cols = col_info.get("group_cols") or []
if isinstance(group_cols, str):
    group_cols = [group_cols]

required = [x_col, y_col]
missing_required = [c for c in required if c is None or c not in df_raw.columns]
if missing_required:
    raise ValueError(f"Missing required coordinate columns: {missing_required}")

print(f"Dataset: {dataset_name}")
print(f"Rows: {len(df_raw):,}")
print(f"Group columns: {group_cols if group_cols else '[single scanpath]'}")
print(f"x/y/t/duration: {x_col}, {y_col}, {t_col}, {duration_col}")

display(df_raw.head())

## Core optimized implementation

The functions below are intentionally independent of the transformer classes. This avoids repeated `fit_transform` calls and makes the timing isolate the feature-family arithmetic after cache construction.

In [ ]:
EPS = 1e-12
LN2 = float(np.log(2.0))


def fmt_param(v: float | int) -> str:
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    if np.isinf(v):
        return "inf"
    return f"{float(v):.6g}".replace("-", "m").replace(".", "p")


def log_base(x, base: float = 2.0):
    if base == 2.0:
        return np.log(x) / LN2
    return np.log(x) / np.log(base)


def as_prob_from_counts(counts: np.ndarray, smoothing: float = 0.0) -> np.ndarray:
    c = np.asarray(counts, dtype=float).ravel()
    if smoothing > 0:
        c = c + float(smoothing)
    total = float(np.sum(c))
    if total <= 0:
        return np.array([], dtype=float)
    return np.clip(c / (total + EPS), EPS, 1.0)


def safe_prob(p: np.ndarray) -> np.ndarray:
    p = np.asarray(p, dtype=float).ravel()
    s = float(np.sum(p))
    if s <= 0:
        return np.array([], dtype=float)
    return np.clip(p / (s + EPS), EPS, 1.0)


def escort_distribution(p: np.ndarray, gamma: float) -> np.ndarray:
    p = safe_prob(p)
    if p.size == 0:
        return p
    return safe_prob(p ** float(gamma))


def tsallis_entropy(p: np.ndarray, q: float, base: float = 2.0) -> float:
    p = np.asarray(p, dtype=float)
    p = p[p > EPS]
    if p.size == 0:
        return 0.0
    q = float(q)
    if np.isclose(q, 1.0):
        return float(-np.sum(p * log_base(p + EPS, base)))
    s = float(np.sum(p ** q))
    return float((1.0 - s) / (q - 1.0) / np.log(base))


def q_log(x: np.ndarray | float, q: float, base: float = 2.0):
    q = float(q)
    if np.isclose(q, 1.0):
        return log_base(x, base)
    return (np.asarray(x) ** (1.0 - q) - 1.0) / ((1.0 - q) * np.log(base))


def q_exp(u: np.ndarray | float, q: float):
    q = float(q)
    if np.isclose(q, 1.0):
        return np.exp(u)
    z = 1.0 + (1.0 - q) * np.asarray(u)
    return np.where(z > 0, z ** (1.0 / (1.0 - q)), 0.0)


def resolve_bounds(coords: np.ndarray, bounds: str | tuple[float, float, float, float]):
    if bounds == "unit":
        xmin, xmax, ymin, ymax = 0.0, 1.0, 0.0, 1.0
    elif bounds == "minmax":
        xmin, xmax = float(np.nanmin(coords[:, 0])), float(np.nanmax(coords[:, 0]))
        ymin, ymax = float(np.nanmin(coords[:, 1])), float(np.nanmax(coords[:, 1]))
    else:
        xmin, xmax, ymin, ymax = map(float, bounds)
    if np.isclose(xmin, xmax):
        xmax = xmin + 1.0
    if np.isclose(ymin, ymax):
        ymax = ymin + 1.0
    return xmin, xmax, ymin, ymax


def discretize_spatial_states(coords: np.ndarray, grid_size: int, bounds: str | tuple[float, float, float, float] = "minmax") -> np.ndarray:
    coords = np.asarray(coords, dtype=float)
    xmin, xmax, ymin, ymax = resolve_bounds(coords, bounds)
    x = np.clip(coords[:, 0], xmin, xmax)
    y = np.clip(coords[:, 1], ymin, ymax)
    xn = (x - xmin) / (xmax - xmin + EPS)
    yn = (y - ymin) / (ymax - ymin + EPS)
    xi = np.minimum((xn * grid_size).astype(int), grid_size - 1)
    yi = np.minimum((yn * grid_size).astype(int), grid_size - 1)
    return (xi * grid_size + yi).astype(int)


def state_counts(states: np.ndarray, n_states: int) -> np.ndarray:
    return np.bincount(np.asarray(states, dtype=int), minlength=int(n_states)).astype(float)


def weighted_state_counts(states: np.ndarray, n_states: int, weights: np.ndarray) -> np.ndarray:
    states = np.asarray(states, dtype=int)
    weights = np.asarray(weights, dtype=float)
    out = np.zeros(int(n_states), dtype=float)
    valid = (states >= 0) & (states < n_states)
    np.add.at(out, states[valid], weights[valid])
    return out


def transition_counts(states: np.ndarray, n_states: int, weights: np.ndarray | None = None) -> np.ndarray:
    states = np.asarray(states, dtype=int)
    out = np.zeros(int(n_states) * int(n_states), dtype=float)
    if states.size < 2:
        return out.reshape(int(n_states), int(n_states))
    i = states[:-1]
    j = states[1:]
    valid = (i >= 0) & (i < n_states) & (j >= 0) & (j < n_states)
    idx = i[valid] * int(n_states) + j[valid]
    if weights is None:
        np.add.at(out, idx, 1.0)
    else:
        w = np.asarray(weights, dtype=float)[:-1][valid]
        np.add.at(out, idx, w)
    return out.reshape(int(n_states), int(n_states))


def transition_matrix_from_counts(C: np.ndarray, smoothing: float = 0.0) -> np.ndarray:
    C = np.asarray(C, dtype=float)
    if smoothing > 0:
        C = C + float(smoothing)
    row_sum = C.sum(axis=1, keepdims=True)
    P = np.zeros_like(C, dtype=float)
    mask = row_sum.squeeze() > 0
    P[mask] = C[mask] / (row_sum[mask] + EPS)
    if smoothing == 0.0:
        empty_rows = np.where(~mask)[0]
        P[empty_rows, empty_rows] = 1.0
    return np.clip(P, EPS, 1.0)


def stationary_distribution(P: np.ndarray, max_iter: int = 500, tol: float = 1e-10) -> np.ndarray:
    n = P.shape[0]
    pi = np.full(n, 1.0 / n, dtype=float)
    for _ in range(max_iter):
        new = pi @ P
        if np.linalg.norm(new - pi, ord=1) < tol:
            return safe_prob(new)
        pi = new
    return safe_prob(pi)


def grid_centroids(grid_size: int, bounds: str | tuple[float, float, float, float] = "unit") -> np.ndarray:
    if bounds == "unit" or bounds == "minmax":
        xmin, xmax, ymin, ymax = 0.0, 1.0, 0.0, 1.0
    else:
        xmin, xmax, ymin, ymax = map(float, bounds)
    xs = np.linspace(xmin, xmax, grid_size + 1)
    ys = np.linspace(ymin, ymax, grid_size + 1)
    xc = 0.5 * (xs[:-1] + xs[1:])
    yc = 0.5 * (ys[:-1] + ys[1:])
    centroids = np.zeros((grid_size * grid_size, 2), dtype=float)
    for xi in range(grid_size):
        for yi in range(grid_size):
            centroids[xi * grid_size + yi] = [xc[xi], yc[yi]]
    return centroids


def transition_reference_Q(grid_size: int, beta: float, bounds: str = "unit") -> np.ndarray:
    cent = grid_centroids(grid_size, bounds=bounds)
    diff = cent[:, None, :] - cent[None, :, :]
    cost = np.sqrt(np.sum(diff * diff, axis=2))
    W = np.exp(-float(beta) * cost)
    return W / (np.sum(W, axis=1, keepdims=True) + EPS)

## Shared cache construction

This is the part that was previously repeated across many feature configurations. Its time is measured separately and is **not** included in marginal Tsallis/thermodynamic times.

In [ ]:
@dataclass
class GridCache:
    grid_size: int
    bounds: str
    states: np.ndarray
    n_states: int
    state_p: np.ndarray
    weighted_state_p: np.ndarray | None
    edge_p: np.ndarray
    weighted_edge_p: np.ndarray | None
    P: np.ndarray
    pi_stationary: np.ndarray
    pi_row_mass: np.ndarray
    surprisal_state_E: np.ndarray
    surprisal_transition_E: np.ndarray


@dataclass
class ScanpathCache:
    index: str
    n_fixations: int
    coords: np.ndarray
    t: np.ndarray | None
    duration: np.ndarray | None
    weights: np.ndarray | None
    energies: dict[str, np.ndarray] = field(default_factory=dict)
    grids: dict[tuple[int, str], GridCache] = field(default_factory=dict)


@dataclass
class DatasetCache:
    dataset_name: str
    group_cols: list[str]
    scanpaths: list[ScanpathCache]
    Q_reference: dict[tuple[int, float], np.ndarray]
    build_seconds: float


def normalize_energy(E: np.ndarray) -> np.ndarray:
    E = np.asarray(E, dtype=float)
    E = E[np.isfinite(E)]
    if E.size == 0:
        return E
    Emin = float(np.min(E))
    Emax = float(np.max(E))
    if np.isclose(Emin, Emax):
        return E - Emin
    return (E - Emin) / (Emax - Emin)


def compute_kinematic_energies(coords: np.ndarray, t: np.ndarray | None, duration: np.ndarray | None) -> dict[str, np.ndarray]:
    energies: dict[str, np.ndarray] = {}
    if coords.shape[0] >= 2:
        v = np.diff(coords, axis=0)
        step = np.linalg.norm(v, axis=1) + EPS
        energies["step_length"] = step
        if t is not None and len(t) == coords.shape[0]:
            dt = np.diff(t) + EPS
            energies["speed"] = step / dt
        if v.shape[0] >= 2:
            v1, v2 = v[:-1], v[1:]
            dot = np.sum(v1 * v2, axis=1)
            n1 = np.linalg.norm(v1, axis=1) + EPS
            n2 = np.linalg.norm(v2, axis=1) + EPS
            cosang = np.clip(dot / (n1 * n2), -1.0, 1.0)
            energies["turning"] = np.arccos(cosang) + EPS
        else:
            energies["turning"] = np.array([], dtype=float)
    else:
        energies["step_length"] = np.array([], dtype=float)
        energies["turning"] = np.array([], dtype=float)
    if duration is not None and len(duration) == coords.shape[0]:
        energies["duration_inv"] = 1.0 / (duration + EPS)
    return energies


def _group_key_to_str(key) -> str:
    if isinstance(key, tuple):
        return "|".join(map(str, key))
    return str(key)


def iter_scanpath_groups(df: pd.DataFrame, group_cols: list[str]):
    if group_cols:
        iterator = df.groupby(group_cols, sort=False, dropna=False)
        for key, g in iterator:
            yield _group_key_to_str(key), g
    else:
        yield "all", df


def build_dataset_cache(
    df: pd.DataFrame,
    dataset_name: str,
    col_info: dict[str, Any],
    grid_sizes: Iterable[int] = GRID_SIZES,
    smoothing: float = SMOOTHING,
    max_scanpaths: int | None = MAX_SCANPATHS,
) -> DatasetCache:
    start = time.perf_counter()

    x_col = col_info.get("x_col")
    y_col = col_info.get("y_col")
    t_col = col_info.get("t_col")
    duration_col = col_info.get("duration_col")
    group_cols = col_info.get("group_cols") or []
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    scanpaths: list[ScanpathCache] = []

    for idx, (scanpath_id, g) in enumerate(iter_scanpath_groups(df, group_cols)):
        if max_scanpaths is not None and idx >= max_scanpaths:
            break
        if SORT_WITHIN_SCANPATH and t_col is not None and t_col in g.columns:
            g = g.sort_values(t_col)

        coords = g[[x_col, y_col]].to_numpy(dtype=float)
        valid = np.isfinite(coords).all(axis=1)
        coords = coords[valid]
        if coords.shape[0] == 0:
            continue

        t = None
        if t_col is not None and t_col in g.columns:
            arr = pd.to_numeric(g[t_col], errors="coerce").to_numpy(dtype=float)[valid]
            if np.isfinite(arr).sum() == arr.size:
                t = arr

        duration = None
        weights = None
        if duration_col is not None and duration_col in g.columns:
            arr = pd.to_numeric(g[duration_col], errors="coerce").to_numpy(dtype=float)[valid]
            arr = np.where(np.isfinite(arr) & (arr > 0), arr, 0.0)
            duration = arr
            weights = arr

        sp = ScanpathCache(
            index=scanpath_id,
            n_fixations=coords.shape[0],
            coords=coords,
            t=t,
            duration=duration,
            weights=weights,
            energies=compute_kinematic_energies(coords, t=t, duration=duration),
        )

        # minmax cache is used by Tsallis state/edge features and surprisal thermodynamic energies.
        # unit cache is used by ThermodynamicTransitionDivergence, matching the previous notebook config.
        for bounds in ("minmax", "unit"):
            for grid_size in grid_sizes:
                n_states = int(grid_size) * int(grid_size)
                states = discretize_spatial_states(coords, int(grid_size), bounds=bounds)

                c_state = state_counts(states, n_states)
                p_state = as_prob_from_counts(c_state, smoothing=smoothing)

                p_weighted_state = None
                if weights is not None:
                    c_w_state = weighted_state_counts(states, n_states, weights)
                    p_weighted_state = as_prob_from_counts(c_w_state, smoothing=smoothing)

                C = transition_counts(states, n_states)
                P = transition_matrix_from_counts(C, smoothing=smoothing)
                p_edge = as_prob_from_counts(C.reshape(-1), smoothing=smoothing)

                p_weighted_edge = None
                if weights is not None:
                    Cw = transition_counts(states, n_states, weights=weights)
                    p_weighted_edge = as_prob_from_counts(Cw.reshape(-1), smoothing=smoothing)

                pi_row_mass = safe_prob(C.sum(axis=1))
                if pi_row_mass.size == 0:
                    pi_row_mass = np.full(n_states, 1.0 / n_states)
                # Stationary distribution is only needed for ThermodynamicTransitionDivergence,
                # which uses the unit-bounds grid in the previous notebook.
                pi_stationary = stationary_distribution(P) if bounds == "unit" else pi_row_mass

                # Surprisal energy arrays use the same smoothed probabilities/matrices.
                s = states.astype(int)
                E_state = -np.log(np.clip(p_state[s], EPS, 1.0))
                if s.size >= 2:
                    E_trans = -np.log(np.clip(P[s[:-1], s[1:]], EPS, 1.0))
                else:
                    E_trans = np.array([], dtype=float)

                sp.grids[(int(grid_size), bounds)] = GridCache(
                    grid_size=int(grid_size),
                    bounds=bounds,
                    states=states,
                    n_states=n_states,
                    state_p=p_state,
                    weighted_state_p=p_weighted_state,
                    edge_p=p_edge,
                    weighted_edge_p=p_weighted_edge,
                    P=P,
                    pi_stationary=pi_stationary,
                    pi_row_mass=pi_row_mass,
                    surprisal_state_E=E_state,
                    surprisal_transition_E=E_trans,
                )
        scanpaths.append(sp)

    # Q_beta does not depend on scanpath, only on grid and beta; cache it too.
    all_transition_betas = sorted(set(TRANS_COARSE_BETAS).union(FINE_BETAS))
    Q_reference = {
        (int(g), float(beta)): transition_reference_Q(int(g), float(beta), bounds="unit")
        for g in grid_sizes
        for beta in all_transition_betas
    }

    build_seconds = time.perf_counter() - start
    return DatasetCache(
        dataset_name=dataset_name,
        group_cols=group_cols,
        scanpaths=scanpaths,
        Q_reference=Q_reference,
        build_seconds=build_seconds,
    )


cache = build_dataset_cache(df_raw, dataset_name, col_info)

print(f"Cached scanpaths: {len(cache.scanpaths):,}")
print(f"Total fixations in cached scanpaths: {sum(sp.n_fixations for sp in cache.scanpaths):,}")
print(f"Cache construction time: {cache.build_seconds:.4f} s")

## Marginal Tsallis feature computation

This uses cached probability vectors for state, duration-weighted state, edge, and duration-weighted edge distributions. The only repeated operations here are the actual Tsallis / escort computations over different `q` values.

In [ ]:
def add_tsallis_values(row: dict[str, float], prefix: str, p: np.ndarray, qs: Iterable[float], grid_size: int, escort_gamma: float | None = None):
    if p is None or len(p) == 0:
        return
    p_use = escort_distribution(p, escort_gamma) if escort_gamma is not None else p
    esc_suffix = "" if escort_gamma is None else f"_esc{fmt_param(escort_gamma)}"
    for q in qs:
        row[f"{prefix}_tsallis_q{fmt_param(q)}{esc_suffix}_g{grid_size}"] = tsallis_entropy(p_use, q=q, base=BASE)


def compute_tsallis_features(cache: DatasetCache, qs: Iterable[float]) -> pd.DataFrame:
    rows = []
    index = []
    qs = tuple(float(q) for q in qs)

    for sp in cache.scanpaths:
        row: dict[str, float] = {}
        for grid_size in GRID_SIZES:
            gc_minmax = sp.grids[(int(grid_size), "minmax")]

            # State occupancy: unweighted and duration-weighted.
            add_tsallis_values(row, "state_entropy", gc_minmax.state_p, qs, grid_size, escort_gamma=None)
            add_tsallis_values(row, "state_entropy", gc_minmax.state_p, qs, grid_size, escort_gamma=0.5)
            add_tsallis_values(row, "state_entropy", gc_minmax.state_p, qs, grid_size, escort_gamma=2.0)

            if gc_minmax.weighted_state_p is not None:
                add_tsallis_values(row, "state_entropy_w_duration", gc_minmax.weighted_state_p, qs, grid_size, escort_gamma=None)
                add_tsallis_values(row, "state_entropy_w_duration", gc_minmax.weighted_state_p, qs, grid_size, escort_gamma=0.5)
                add_tsallis_values(row, "state_entropy_w_duration", gc_minmax.weighted_state_p, qs, grid_size, escort_gamma=2.0)

            # Edge occupancy: unweighted and duration-weighted.
            add_tsallis_values(row, "edge_entropy", gc_minmax.edge_p, qs, grid_size, escort_gamma=None)
            if gc_minmax.weighted_edge_p is not None:
                add_tsallis_values(row, "edge_entropy_duration", gc_minmax.weighted_edge_p, qs, grid_size, escort_gamma=None)

        rows.append(row)
        index.append(sp.index)

    return pd.DataFrame(rows, index=index)

## Marginal thermodynamic feature computation

Energy series and grid-based surprisal energy series are already cached. This section measures only the thermodynamic statistics over `beta` / `q_ensemble` values and the cached transition-divergence matrices.

In [ ]:
def thermo_boltzmann_observables(E: np.ndarray, betas: Iterable[float]) -> list[tuple[float, float, float, float]]:
    En = normalize_energy(E)
    if En.size == 0:
        return [(0.0, 0.0, 0.0, 0.0) for _ in betas]
    out = []
    for beta in betas:
        beta = float(beta)
        w = np.exp(-beta * En)
        Z = float(np.sum(w)) + EPS
        q = w / Z
        meanE = float(np.sum(q * En))
        F = -(1.0 / beta) * float(log_base(Z, BASE))
        # This matches the original ThermodynamicObservables implementation.
        S = float(log_base(Z, BASE)) + beta * meanE
        varE = float(np.sum(q * (En - meanE) ** 2))
        C = beta ** 2 * varE
        out.append((float(F), float(meanE), float(S), float(C)))
    return out


def thermo_tsallis_ensemble_observables(E: np.ndarray, betas: Iterable[float], q_ensemble: float) -> list[tuple[float, float, float, float]]:
    En = normalize_energy(E)
    if En.size == 0:
        return [(0.0, 0.0, 0.0, 0.0) for _ in betas]
    out = []
    for beta in betas:
        beta = float(beta)
        u = q_exp(-beta * En, q=q_ensemble)
        u = np.where(np.isfinite(u) & (u > 0), u, 0.0)
        Z = float(np.sum(u))
        if Z <= 0:
            w = np.full_like(En, 1.0 / max(En.size, 1), dtype=float)
        else:
            w = u / Z
        Em = float(np.sum(w * En))
        Var = float(np.sum(w * (En - Em) ** 2))
        C = beta ** 2 * Var
        S = tsallis_entropy(w, q=q_ensemble, base=BASE)
        F = -(1.0 / max(beta, EPS)) * float(q_log(max(Z, EPS), q=q_ensemble, base=BASE))
        out.append((float(F), float(Em), float(S), float(C)))
    return out


def add_observable_columns(row: dict[str, float], prefix: str, stats: list[tuple[float, float, float, float]], betas: Iterable[float], suffix: str):
    for beta, (F, Em, S, C) in zip(betas, stats):
        bb = fmt_param(beta)
        row[f"{prefix}_F_b{bb}_{suffix}"] = F
        row[f"{prefix}_E_b{bb}_{suffix}"] = Em
        row[f"{prefix}_S_b{bb}_{suffix}"] = S
        row[f"{prefix}_C_b{bb}_{suffix}"] = C


def transition_divergence_kl(gc_unit: GridCache, Q: np.ndarray) -> float:
    P = gc_unit.P
    D_rows = np.sum(P * log_base(P / (Q + EPS), BASE), axis=1)
    return float(np.sum(gc_unit.pi_stationary * D_rows))


def available_energy_names(cache: DatasetCache) -> list[str]:
    names = ["step_length", "turning"]
    if any("speed" in sp.energies for sp in cache.scanpaths):
        names.append("speed")
    if any("duration_inv" in sp.energies for sp in cache.scanpaths):
        names.append("duration_inv")
    return names


def get_energy_series(sp: ScanpathCache, energy: str, grid_size: int | None = None) -> np.ndarray:
    if energy in sp.energies:
        return sp.energies[energy]
    if energy == "surprisal_state":
        return sp.grids[(int(grid_size), "minmax")].surprisal_state_E
    if energy == "surprisal_transition":
        return sp.grids[(int(grid_size), "minmax")].surprisal_transition_E
    return np.array([], dtype=float)


def compute_thermodynamic_fine_features(cache: DatasetCache) -> pd.DataFrame:
    rows = []
    index = []
    non_grid_energies = available_energy_names(cache)

    for sp in cache.scanpaths:
        row: dict[str, float] = {}

        for q_ens in Q_ENSEMBLES_FINE:
            q_suffix = f"tsallis_q{fmt_param(q_ens)}"

            for energy in non_grid_energies:
                stats = thermo_tsallis_ensemble_observables(get_energy_series(sp, energy), FINE_BETAS, q_ensemble=q_ens)
                add_observable_columns(row, "thermo2", stats, FINE_BETAS, suffix=f"{energy}_{q_suffix}")

            for grid_size in GRID_SIZES:
                for energy in ("surprisal_state", "surprisal_transition"):
                    stats = thermo_tsallis_ensemble_observables(get_energy_series(sp, energy, grid_size=grid_size), FINE_BETAS, q_ensemble=q_ens)
                    add_observable_columns(row, "thermo2", stats, FINE_BETAS, suffix=f"{energy}_{q_suffix}_g{grid_size}")

        # Fine transition divergence battery.
        for grid_size in GRID_SIZES:
            gc_unit = sp.grids[(int(grid_size), "unit")]
            for beta in FINE_BETAS:
                Q = cache.Q_reference[(int(grid_size), float(beta))]
                row[f"thermo_trans_div_kl_b{fmt_param(beta)}_spatial_spatial_g{grid_size}"] = transition_divergence_kl(gc_unit, Q)

        rows.append(row)
        index.append(sp.index)
    return pd.DataFrame(rows, index=index)

## Benchmark helper

In [ ]:
def benchmark_feature_function(name: str, family: str, func, repeats: int = N_REPEATS):
    times = []
    last_df = None
    for r in range(repeats):
        gc.collect()
        t0 = time.perf_counter()
        last_df = func()
        elapsed = time.perf_counter() - t0
        times.append(elapsed)
    assert last_df is not None
    n_rows, n_features = last_df.shape
    mean_s = float(np.mean(times))
    std_s = float(np.std(times, ddof=1)) if len(times) > 1 else 0.0
    return last_df, {
        "battery": name,
        "family": family,
        "n_repeats": repeats,
        "n_scanpaths": n_rows,
        "n_features": n_features,
        "mean_seconds": mean_s,
        "std_seconds": std_s,
        "min_seconds": float(np.min(times)),
        "max_seconds": float(np.max(times)),
        "per_scanpath_ms": mean_s / max(n_rows, 1) * 1_000.0,
        "per_feature_value_us": mean_s / max(n_rows * n_features, 1) * 1_000_000.0,
        "cache_build_seconds_not_included": cache.build_seconds,
        "one_pass_total_seconds_cache_plus_mean": cache.build_seconds + mean_s,
    }

## Run marginal timing

In [ ]:
benchmarks = []
feature_tables = {}


feature_tables["tsallis_fine"], row = benchmark_feature_function(
    "tsallis_fine", "tsallis", lambda: compute_tsallis_features(cache, FINE_QS)
)
benchmarks.append(row)


feature_tables["thermodynamic_fine"], row = benchmark_feature_function(
    "thermodynamic_fine", "thermodynamic", lambda: compute_thermodynamic_fine_features(cache)
)
benchmarks.append(row)

summary = pd.DataFrame(benchmarks)
summary.insert(0, "dataset", cache.dataset_name)
summary.insert(1, "total_fixations", sum(sp.n_fixations for sp in cache.scanpaths))
summary

## Compact result table

In [ ]:
compact_cols = [
    "dataset", "family", "battery", "n_scanpaths", "total_fixations", "n_features",
    "mean_seconds", "std_seconds", "per_scanpath_ms", "per_feature_value_us",
    "cache_build_seconds_not_included", "one_pass_total_seconds_cache_plus_mean",
]
summary_compact = summary[compact_cols].copy()
for c in ["mean_seconds", "std_seconds", "per_scanpath_ms", "per_feature_value_us", "cache_build_seconds_not_included", "one_pass_total_seconds_cache_plus_mean"]:
    summary_compact[c] = summary_compact[c].astype(float)

display(summary_compact)